In [24]:
IMAGE_OUTPUT = "data/dataset/images/sam3_images"
EXAMPLE_IMAGE = "data/dataset/dummy_images/imagenReal.jpg"
MODEL_PATH = "data/dataset/sam3_model/sam3.pt"

In [10]:
from transformers import Sam3Processor, Sam3Model
import torch
from PIL import Image
import requests
# hay que importar python -m pip install -U pip huggingface_hub y logearse con tu token
device = "cuda" if torch.cuda.is_available() else "cpu"

model = Sam3Model.from_pretrained("facebook/sam3").to(device)
processor = Sam3Processor.from_pretrained("facebook/sam3")

# Load image
image_url = "http://images.cocodataset.org/val2017/000000077595.jpg"
image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")

# Segment using text prompt
inputs = processor(images=image, text="ear", return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

# Post-process results
results = processor.post_process_instance_segmentation(
    outputs,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=inputs.get("original_sizes").tolist()
)[0]

print(f"Found {len(results['masks'])} objects")
# Results contain:
# - masks: Binary masks resized to original image size
# - boxes: Bounding boxes in absolute pixel coordinates (xyxy format)
# - scores: Confidence scores


Loading weights: 100%|██████████| 1468/1468 [00:00<00:00, 18109.90it/s]


Found 2 objects


In [34]:
def procesar_directorio_completo(directorio_raiz_imagenes, directorio_salida_etiquetas, model, processor, device):
    """
    Recorre todas las carpetas buscando imágenes y lanza la inferencia con SAM 3.
    """
    for root, dirs, files in os.walk(directorio_raiz_imagenes):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                ruta_imagen = os.path.join(root, file)

                print(f"🔍 Analizando: {ruta_imagen}")
                etiquetar_imagen_con_sam(str(ruta_imagen), directorio_salida_etiquetas, model, processor, device)


In [40]:
import torch
from transformers import Sam3Processor, Sam3Model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Iniciando proceso en: {device}")

# 2. Cargar el modelo y procesador SAM 3 (¡Solo una vez!)
print("⏳ Cargando el modelo SAM 3...")
model = Sam3Model.from_pretrained("facebook/sam3").to(device)
processor = Sam3Processor.from_pretrained("facebook/sam3")
print("✅ Modelo cargado correctamente.")

DIRECTORIO_IMAGENES = "../data/dataset/images/sam3"
DIRECTORIO_ETIQUETAS = "../data/dataset/labels/sam3"

procesar_directorio_completo(DIRECTORIO_IMAGENES, DIRECTORIO_ETIQUETAS, model, processor, device)

print("LO HEMOS CONSEGUIDO VAMOOOOSOS")

🚀 Iniciando proceso en: cuda
⏳ Cargando el modelo SAM 3...


Loading weights: 100%|██████████| 1468/1468 [00:00<00:00, 9191.05it/s]


✅ Modelo cargado correctamente.
🔍 Analizando: ../data/dataset/images/sam3\cogolloEnfermo_ricos_1.jpg
Etiquetas guardadas en: ../data/dataset/labels/sam3\cogolloEnfermo_ricos_1.txt
🔍 Analizando: ../data/dataset/images/sam3\hojaEnferma_ricos_1.jpg
Etiquetas guardadas en: ../data/dataset/labels/sam3\hojaEnferma_ricos_1.txt
🔍 Analizando: ../data/dataset/images/sam3\hojaEnferma_ricos_2.jpg
Etiquetas guardadas en: ../data/dataset/labels/sam3\hojaEnferma_ricos_2.txt
🔍 Analizando: ../data/dataset/images/sam3\hojaEnferma_ricos_3.jpg
Etiquetas guardadas en: ../data/dataset/labels/sam3\hojaEnferma_ricos_3.txt
🔍 Analizando: ../data/dataset/images/sam3\pimientoEnfermo_ricos_1.jpg
Etiquetas guardadas en: ../data/dataset/labels/sam3\pimientoEnfermo_ricos_1.txt
🔍 Analizando: ../data/dataset/images/sam3\pimientoEnfermo_ricos_2.jpg
Etiquetas guardadas en: ../data/dataset/labels/sam3\pimientoEnfermo_ricos_2.txt
🔍 Analizando: ../data/dataset/images/sam3\pimientoEnfermo_ricos_3.jpg
Etiquetas guardadas en: 

In [39]:
import torch

def etiquetar_imagen_con_sam(ruta_imagen, directorio_etiquetas, model, processor, device):
    """
    Lee una imagen, detecta hojas con SAM 3 y guarda las etiquetas en formato YOLO.
    Args:
        ruta_imagen (str): Ruta al archivo de imagen a procesar.
        directorio_etiquetas (str): Ruta al directorio donde se guardarán las etiquetas en formato YOLO.
        model: Modelo SAM 3 cargado.
        processor: Procesador SAM 3 cargado.
        device: Dispositivo para la inferencia (CPU o GPU).
    """
    image = Image.open(ruta_imagen).convert("RGB")

    image.thumbnail((1024, 1024)) # redimensiono porqie si no la grafica se queda sin memoria para procesar la imagen. Esta redimension no importa para los labels ya que los convertimos en formato yolo que hace uso del ancho y alto, ademas los centros se definen entre 0 y 1, porlo qque no deberia de haber problema con esta conversin

    img_width, img_height = image.size

    inputs = processor(images=image, text="leaf", return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    results = processor.post_process_instance_segmentation(
        outputs,
        threshold=0.5,
        mask_threshold=0.5,
        target_sizes=inputs.get("original_sizes").tolist()
    )[0]

    bboxes_sam = results['boxes'].tolist()
    bboxes_yolo = []

    # convertimos cada caja al formato de YOLO para guardar los labels
    for bbox in bboxes_sam:
        bbox_yolo = convertir_sam_a_yolo(bbox, img_width, img_height)
        bboxes_yolo.append(bbox_yolo)

    # guardamos las etiquetas si se encontró alguna hoja
    if bboxes_yolo:
        nombre_archivo = os.path.splitext(os.path.basename(ruta_imagen))[0]
        guardar_etiquetas_yolo(
            ruta_original_imagen = ruta_imagen,
            directorio_salida = directorio_etiquetas,
            nombre_archivo = nombre_archivo,
            bboxes_yolo = bboxes_yolo
        )
    else:
        print(f"No se detectaron hojas en: {ruta_imagen}")

In [32]:
def convertir_sam_a_yolo(bbox_sam, img_width, img_height):
    """
    Convierte coordenadas absolutas [x_min, y_min, x_max, y_max]
    al formato normalizado de YOLO [centro_x, centro_y, ancho, alto].
    """
    # Desempaquetamos la lista que nos da SAM 3
    x_min, y_min, x_max, y_max = bbox_sam

    # 1. Calculamos las coordenadas del centro
    centro_x = (x_min + x_max) / 2.0
    centro_y = (y_min + y_max) / 2.0

    # 2. Calculamos el ancho y alto absoluto de la caja
    ancho = x_max - x_min
    alto = y_max - y_min

    # 3. Normalizamos dividiendo por el tamaño total de la imagen
    centro_x_norm = centro_x / img_width
    centro_y_norm = centro_y / img_height
    ancho_norm = ancho / img_width
    alto_norm = alto / img_height

    # Devolvemos los valores redondeados a 6 decimales (suficiente precisión para YOLO)
    return [round(centro_x_norm, 6), round(centro_y_norm, 6), round(ancho_norm, 6), round(alto_norm, 6)]

In [31]:
def guardar_etiquetas_yolo(ruta_original_imagen, directorio_salida, nombre_archivo, bboxes_yolo, clase_id=0):
    """
    Guarda coordenadas en formato YOLO. Si es un frame, respeta la subcarpeta original.
    """
    ruta_carpeta_origen = os.path.dirname(ruta_original_imagen)
    nombre_carpeta_origen = os.path.basename(ruta_carpeta_origen)

    # Si el archivo empieza por "frame", añadimos esa subcarpeta al destino para que la estructura de images y labels sea la misma, lo que facilita el entrenamiento con YOLO
    if nombre_archivo.startswith('frame'):
        directorio_salida = os.path.join(directorio_salida, nombre_carpeta_origen)

    os.makedirs(directorio_salida, exist_ok=True)

    ruta_archivo = os.path.join(directorio_salida, f"{nombre_archivo}.txt")
    with open(ruta_archivo, 'w') as archivo:
        for bbox in bboxes_yolo:
            cx, cy, w, h = bbox
            linea = f"{clase_id} {cx} {cy} {w} {h}\n"
            archivo.write(linea)

    print(f"Etiquetas guardadas en: {ruta_archivo}")

In [17]:
import cv2
import numpy as np

def extraer_fotogramas_representativos(video_path, output_path, threshold=30.0):
    """
    Extrae fotogramas de un vídeo basándose en la diferencia visual.
    Args:
        video_path (str): Ruta al archivo de vídeo.
        output_path (str): Directorio donde se guardarán los fotogramas extraídos.
        threshold (float): Umbral de diferencia para considerar un fotograma como representativo.
    """
    # Creamos el directorio de salida si no existe
    os.makedirs(output_path, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error al abrir el vídeo: {video_path}")
        return

    # Leemos el primer fotograma
    ret, prev_frame = cap.read()
    if not ret:
        print("El vídeo está vacío o no se puede leer.")
        return

    saved_count = 0

    # Guardamos el primer fotograma por defecto
    cv2.imwrite(os.path.join(output_path, f"frame_{saved_count:04d}.jpg"), prev_frame)
    saved_count += 1

    # Convertimos a escala de grises para la comparación
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Calculamos la diferencia absoluta entre el fotograma actual y el último que analicemos
        diff = cv2.absdiff(prev_gray, gray)
        mean_diff = np.mean(diff)

        # Si la diferencia supera el umbral, consideramos el fotograma representativo
        if mean_diff > threshold:
            cv2.imwrite(os.path.join(output_path, f"frame_{saved_count:04d}.jpg"), frame)
            prev_gray = gray
            saved_count += 1

    cap.release()
    print(f"Proceso completado. Se han extraído {saved_count} imágenes representativas.")


In [14]:
from PIL import Image
import pillow_heif

pillow_heif.register_heif_opener()

def convertir_heic_a_jpg(ruta_heic, ruta_salida_jpg):
    """
    Lee un archivo HEIC y lo guarda como JPG.
    Args:
        ruta_heic (str): Ruta al archivo HEIC de entrada.
        ruta_salida_jpg (str): Ruta donde se guardará el archivo JPG convertido.
    """
    try:
        imagen = Image.open(ruta_heic)
        imagen_rgb = imagen.convert('RGB')

        imagen_rgb.save(ruta_salida_jpg, format="JPEG")
        print(f"✅ Imagen convertida y guardada en: {ruta_salida_jpg}")

    except Exception as e:
        print(f"❌ Error al procesar la imagen {ruta_heic}: {e}")

In [28]:
import os
destination = "pre-trainingsetSam3/processed"
source = "pre-trainingsetSam3/non-processed"
print(f"Archivos de salida: {os.listdir(destination)}")
print(f"Archivos de salida: {os.listdir(source)}")
for root, dirs, files in os.walk(source):
    for file in files:
        extension = file.lower()
        file_path = os.path.join(root, file)

        if extension.endswith('.mov'):
            print(f"🎬 Procesando vídeo: {file}")

            # creamos subcarpetas para tener mas organizadas las fotos de los videos
            nombre_video = os.path.splitext(file)[0]
            ruta_salida_video = os.path.join(destination, nombre_video)

            # Llamamos a la función que creamos en el paso anterior
            extraer_fotogramas_representativos(str(file_path), str(ruta_salida_video), threshold=50.0)

        elif extension.endswith('.heic'):
            print(f"📸 Archivo HEIC encontrado: {file_path}")
            nombre_foto = os.path.splitext(file)[0]
            ruta_salida_jpg = os.path.join(destination, nombre_foto+".jpg")
            convertir_heic_a_jpg(str(file_path), str(ruta_salida_jpg))

Archivos de salida: []
Archivos de salida: ['Enfermas', 'Lineos sanos']
📸 Archivo HEIC encontrado: pre-trainingsetSam3/non-processed\Enfermas\cogolloEnfermo_ricos_1.HEIC
✅ Imagen convertida y guardada en: pre-trainingsetSam3/processed\cogolloEnfermo_ricos_1.jpg
🎬 Procesando vídeo: cogollosEnfermos_ricos_1.mov
Proceso completado. Se han extraído 87 imágenes representativas.
📸 Archivo HEIC encontrado: pre-trainingsetSam3/non-processed\Enfermas\hojaEnferma_ricos_1.HEIC
✅ Imagen convertida y guardada en: pre-trainingsetSam3/processed\hojaEnferma_ricos_1.jpg
📸 Archivo HEIC encontrado: pre-trainingsetSam3/non-processed\Enfermas\hojaEnferma_ricos_2.HEIC
✅ Imagen convertida y guardada en: pre-trainingsetSam3/processed\hojaEnferma_ricos_2.jpg
📸 Archivo HEIC encontrado: pre-trainingsetSam3/non-processed\Enfermas\hojaEnferma_ricos_3.HEIC
✅ Imagen convertida y guardada en: pre-trainingsetSam3/processed\hojaEnferma_ricos_3.jpg
📸 Archivo HEIC encontrado: pre-trainingsetSam3/non-processed\Enfermas\pi